# Lab Identification — Lab-ID Mode

Interactive notebook with **live streaming HTML** that mimics the `/refine_with_kg` route.
Agent reasoning, tool calls, tool results, and recommendations render in real-time
using the same `chat_med_refinement.html` UI.

## 0. Setup

In [ ]:
import sys, os, re, json, html as html_mod, uuid
from collections import defaultdict
from typing import List

NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

from IPython.display import display, HTML, update_display

os.environ["IMO_NORMALIZE_CLIENT_ID"] = ""
os.environ["IMO_NORMALIZE_SECRET"]    = ""
os.environ["IMO_KG_CLIENT_ID"]        = ""
os.environ["IMO_KG_CLIENT_SECRET"]    = ""

os.environ["AWS_ACCESS_KEY_ID"]     = ""
os.environ["AWS_SECRET_ACCESS_KEY"] = ""
os.environ["AWS_SESSION_TOKEN"]     = ""
os.environ["AWS_DEFAULT_REGION"]    = "us-east-1"

import kg_config as kc
print("\u2713  Credentials loaded")

✓  Credentials loaded


## 1. LLM, Tools & System Prompt

In [2]:
from langchain_aws import ChatBedrockConverse
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from kg_api_client import KGApiClient
from typing import Any

llm = ChatBedrockConverse(
    model_id=os.getenv("MODEL_ID", "us.anthropic.claude-haiku-4-5-20251001-v1:0"),
    region_name=os.getenv("AWS_REGION", "us-east-1"),
    temperature=0,
    provider="anthropic",
    max_tokens=64000,
)
client = KGApiClient()

# --- Token overflow prevention (from route's _cap_tool_response) ---
_KG_LIST_THRESHOLD = 200
_MAX_TOOL_RESPONSE_CHARS = 100000

def _truncate_kg_lists(data, candidate_problems=None):
    if not isinstance(data, dict):
        return data
    list_fields = [
        "treatedProblems", "causedProblems", "domainNarrower", "narrower",
        "synonyms", "associatedTreatments", "contraindicatedTreatments",
        "supportiveTreatments", "domainNarrowerByMedications",
        "domainNarrowerByProcedures", "containingMedications",
    ]
    result = {}
    for key, value in data.items():
        if key in list_fields and isinstance(value, list) and len(value) > _KG_LIST_THRESHOLD:
            total_count = len(value)
            if candidate_problems:
                candidates_lower = [c.lower().strip() for c in candidate_problems]
                matched = [
                    item for item in value
                    if any(
                        cand in item.get("title", "").lower()
                        or item.get("title", "").lower() in cand
                        for cand in candidates_lower
                    )
                ]
                result[key] = matched
                result[key + "_summary"] = (
                    "KG returned " + str(total_count) + " total items. "
                    + "Showing " + str(len(matched)) + " items matching candidates. "
                    + "Full list too large (" + str(total_count) + " items)."
                )
            else:
                result[key] = value[:_KG_LIST_THRESHOLD]
                result[key + "_summary"] = (
                    "KG returned " + str(total_count) + " total items. "
                    + "Showing first " + str(_KG_LIST_THRESHOLD) + " of " + str(total_count) + ". "
                    + "Truncated to prevent token overflow."
                )
        elif key == "lexical" and isinstance(value, dict):
            result[key] = _truncate_kg_lists(value, candidate_problems)
        else:
            result[key] = value
    return result

def _aggressive_truncate(data, max_items):
    if not isinstance(data, dict):
        return data
    result = {}
    for key, value in data.items():
        if isinstance(value, list) and len(value) > max_items:
            total = len(value)
            result[key] = value[:max_items]
            result[key + "_summary"] = (
                "KG returned " + str(total) + " total items. "
                + "Showing first " + str(max_items) + " of " + str(total)
                + " (truncated for token limit)."
            )
        elif isinstance(value, dict):
            result[key] = _aggressive_truncate(value, max_items)
        else:
            result[key] = value
    return result

def _cap_tool_response(result, candidate_problems=None):
    result = _truncate_kg_lists(result, candidate_problems)
    output = json.dumps(result, indent=2)
    if len(output) <= _MAX_TOOL_RESPONSE_CHARS:
        return output
    shrink_limit = 50
    while len(output) > _MAX_TOOL_RESPONSE_CHARS and shrink_limit >= 10:
        result = _aggressive_truncate(result, shrink_limit)
        output = json.dumps(result, indent=2)
        shrink_limit = shrink_limit // 2
    if len(output) > _MAX_TOOL_RESPONSE_CHARS:
        result = _aggressive_truncate(result, 5)
        output = json.dumps(result, indent=2)
    return output

# --- Tool definitions ---

@tool
def normalize_medical_term(input_term: str, domain: str = "Lab", order_name: str = "") -> str:
    """Normalize a medical term using IMO Precision Normalize API.
    Args:
        input_term: The clinical term (e.g., \"glucose\", \"hemoglobin A1c\", \"lipid panel\"). Must not be empty.
        domain: \"Medication\" for drugs, \"Problem\" for diagnoses, \"Lab\" for lab results. Default is Lab.
        order_name: Optional lab order name (e.g., \"lipid panel\", \"comprehensive metabolic panel\"). Only used when domain=\"Lab\". If you only have the order name and not the component name, pass the order name as input_term instead.
    """
    if not input_term or not input_term.strip():
        return json.dumps({"success": False, "error": "STOP: input_term is empty. Do NOT retry with empty string."})
    result = client.normalize_medical_term(input_term.strip(), domain, order_name=order_name.strip() if order_name else "")
    return json.dumps(result, indent=2)

@tool
def identify_lab(lab_code: str) -> str:
    """Query the KG for a lab's narrower concepts and LOINC code mappings.

    Given an IMO lexical code for a lab (from normalize_medical_term with domain=\"Lab\"),
    queries the KG procedure domain to get domainNarrower concepts and their LOINC codes.

    Args:
        lab_code: The IMO lexical code of the lab (from normalize_medical_term with domain=\"Lab\"). REQUIRED.
    """
    if not lab_code or not lab_code.strip():
        return json.dumps({"success": False, "error": "STOP: lab_code is empty. Normalize the lab first with domain='Lab'."})
    result = client.identify_lab(lab_code.strip())
    return _cap_tool_response(result)

LAB_ID_TOOLS = [normalize_medical_term, identify_lab]

SYSTEM_PROMPT = """You are a lab identification agent. Given a clinical note containing laboratory orders, you normalize each lab using the IMO Precision Normalize API, then query the IMO Knowledge Graph to find the IMO description and LOINC code for each lab.

## WORKFLOW

### Step 1: Extract and Classify Labs (TEXT ONLY \u2014 NO TOOL CALLS)
Read the clinical note and extract labs ONLY from the laboratory/lab orders section (e.g., \"Laboratory observations ordered today\", \"Lab orders\", etc.). Do NOT extract items from evaluation procedures, imaging, or other non-lab sections \u2014 those are procedures, not labs.

For each lab, you MUST:

**A. Classify as ORDER or COMPONENT:**
- **ORDER** = a panel or test order that groups multiple analytes (e.g., \"comprehensive metabolic panel\", \"lipid panel\", \"complete blood count\", \"basic metabolic panel\", \"hepatic function panel\", \"thyroid panel\"). These are requests to perform a set of tests.
- **COMPONENT** = a single analyte or result being measured (e.g., \"glucose\", \"sodium\", \"cholesterol total\", \"serum prealbumin\", \"hemoglobin A1c\", \"urine protein\"). These are individual measurable values.

**Classification rules:**
- If the item is a well-known panel name \u2192 classify as ORDER
- If the item is a single measurable analyte/substance \u2192 classify as COMPONENT
- If a component is listed under or associated with a panel in the note, record the panel as its order_name
- If unsure, default to COMPONENT (safer for normalization)

**B. Note clinical context clues:**
- Specimen type (serum, urine, CSF, plasma, blood, etc.)
- Collection method (spot, timed 24-hour, random, etc.)
- Measurement type (quantitative, qualitative, ratio, panel, etc.)
- Units, method, or local order codes if mentioned

Output a structured summary with classification reasoning. Example:
\"Extracted labs (from Laboratory observations section):

| # | Lab Term | Classification | Reasoning | order_name | Context Clues |
|---|---|---|---|---|---|
| 1 | Urine protein, quantitative | COMPONENT | Single analyte (protein) being measured | \u2014 | specimen: urine, method: timed 24-hour, measurement: quantitative |
| 2 | Serum prealbumin | COMPONENT | Single analyte (prealbumin) being measured | \u2014 | specimen: serum, local order code: PALB U |
| 3 | Comprehensive metabolic panel | ORDER | Panel grouping multiple analytes (glucose, electrolytes, etc.) | \u2014 | typically serum |
| 4 | Glucose | COMPONENT | Single analyte from CMP | comprehensive metabolic panel | specimen: serum |
| 5 | Lipid panel | ORDER | Panel grouping cholesterol, triglycerides, etc. | \u2014 | typically serum |\"

DO NOT call any tool during this step.

### Step 2: Normalize All Labs

**Before making any tool calls**, output a reasoning table showing exactly how you will call normalize for each lab based on the Step 1 classification:

#### Normalize Plan

| # | Lab Term | Type | input_term | order_name | Reasoning |
|---|---|---|---|---|---|
| 1 | (term from note) | ORDER/COMPONENT | (value to pass) | (value to pass or \u2014) | (why this classification and these values) |

Example:
| # | Lab Term | Type | input_term | order_name | Reasoning |
|---|---|---|---|---|---|
| 1 | Glucose | COMPONENT | glucose | comprehensive metabolic panel | Single analyte; listed under CMP in the note |
| 2 | Comprehensive metabolic panel | ORDER | comprehensive metabolic panel | \u2014 | Panel grouping multiple analytes; no separate component name available |
| 3 | Serum prealbumin | COMPONENT | Serum prealbumin | \u2014 | Single analyte; not part of any panel in the note |

Then call normalize_medical_term for each lab using these rules:

**For COMPONENT labs:**
- Pass the component/analyte name as input_term
- If the component belongs to a panel identified in Step 1, pass the panel name as order_name
- normalize_medical_term(input_term=<component name>, domain=\"Lab\", order_name=<panel name if known>)

**For ORDER labs (panels):**
- Pass the order/panel name as input_term (since there is no separate component name)
- normalize_medical_term(input_term=<order name>, domain=\"Lab\")

CRITICAL RULES:
- ALWAYS use domain=\"Lab\" for lab normalization.
- NEVER pass an empty string as input_term.
- If normalization returns no matches, list that lab as \"Could not normalize\" in the final output and skip it.
- Record each lab's normalized title, default_lexical_code, and loinc_codes from the normalization response.

After normalizing all labs, present a **Lab Normalization Results** table:

#### Lab Normalization Results

| # | Input Term | IMO Title | IMO Code | LOINC Codes | Score |
|---|---|---|---|---|---|
| 1 | (input term) | (lexical_title) | (default_lexical_code) | (comma-separated LOINC codes from loinc_codes) | (score) |

Then show the **Raw Normalization Response** in a collapsed details block:
<details><summary>Raw Normalization Response</summary>

```json
(paste the raw_response from the normalize result here)
```
</details>

You may batch multiple normalize calls in parallel.

### Step 3: Query KG for Each Lab
For EACH successfully normalized lab, call identify_lab with its default_lexical_code:
- identify_lab(lab_code=<default_lexical_code>)

This queries the KG procedure domain and returns:
- The base lab concept with its LOINC codes (if any)
- domainNarrower concepts \u2014 more specific lab descriptions, each with their own LOINC codes

You may batch multiple identify_lab calls in parallel.

### Step 4: Select the Best Narrower Concept for Each Lab

For each lab, compare the KG narrower concepts against the clinical context clues from Step 1 to pick the single best match:

**Selection criteria (in priority order):**
1. **Specimen match**: If the note says \"serum\", pick the narrower concept with \"serum\" in the title. If \"urine\", pick the urine variant. Never pick \"urine\" when the note says \"serum\".
2. **Method/collection match**: If the note says \"24-hour collection\", prefer the timed/24-hour variant. If \"spot specimen\", prefer the spot/random variant.
3. **Measurement type match**: If the note says \"quantitative\", prefer the quantitative variant over qualitative.
4. **Most specific with LOINC**: Among remaining candidates, prefer the one that has a LOINC code and is most specific.

If no narrower concept matches the clinical context, use the base concept from normalize.
If the base concept has no narrower concepts at all, use the base concept directly.

### Step 5: Present Results

#### Lab Identification Results

| # | Lab (from Note) | IMO Description | IMO Code | LOINC Code |
|---|---|---|---|---|
| 1 | (original term from note) | (selected IMO title from KG) | (IMO code) | (LOINC codes \u2014 see rules below) |

**LOINC Code column rules \u2014 KG-first, normalize-fallback:**
- **Primary source: KG (identify_lab).** Show ALL LOINC codes from the KG response for the selected concept (`loinc_codes` in the selected narrower concept, or root-level `loinc_codes`), comma-separated. If multiple LOINC codes are available, list all of them \u2014 NEVER show only one when more exist.
- **Fallback source: Normalize.** If the KG returned NO LOINC codes for the selected concept, fall back to the LOINC codes from the Step 2 normalize response (`loinc_codes` field in the match). When using normalize LOINC codes, append \"(from Normalize)\" after the codes to clearly indicate the source (e.g., \"2160-0, 38483-4 (from Normalize)\").
- If neither KG nor normalize has LOINC codes, show \"\u2014\".

The \"IMO Description\" column MUST be populated from the KG result \u2014 either the best-matching narrower concept or the base concept. Never use the raw note term or your own description.

For each lab where you picked a narrower concept over the base, add a brief note explaining why:
> \"Selected 'Serum prealbumin measurement (mass/volume)' over base 'transthyretin measurement' because note specifies specimen: serum.\"

**If multiple narrower concepts are equally valid** (no clear context clue to differentiate), list them as options:

| Option | IMO Description | IMO Code | LOINC Code |
|---|---|---|---|
| A | (variant 1) | (code) | (LOINC codes per rules above) |
| B | (variant 2) | (code) | (LOINC codes per rules above) |

---

**Summary:**
- X labs extracted from clinical note
- Y labs identified with LOINC codes from KG
- Z labs could not be resolved

## ABSOLUTE RULES
- ONLY present data returned by the KG. Never fabricate LOINC codes or IMO descriptions.
- The \"IMO Description\" MUST come from the KG (either base concept or a narrower concept) \u2014 never invent descriptions.
- A lab has a LOINC code ONLY if the KG returns it in the mappings \u2014 do NOT use your own knowledge to assign LOINC codes.
- If identify_lab returns empty results, state \"No data found in KG for this lab\" \u2014 do NOT invent results.
- If normalization fails, report the error and skip that lab.
- Before each tool call, briefly state what you are doing.
- NEVER call normalize_medical_term with an empty string.

## SUMMARY BLOCK (REQUIRED)
After your final answer, include a summary block wrapped in `[SUMMARY]` and `[/SUMMARY]` tags. This is displayed in a visual explainer panel for non-technical audiences. Write in simple, plain language.

**FORMAT: Exactly 4 paragraphs separated by blank lines.** Each paragraph maps to a visual card:

1. **Paragraph 1 \u2014 Clinical Question:** Restate what the user asked in 1-2 plain sentences. Mention the laboratory orders being identified.
2. **Paragraph 2 \u2014 What the Agent Did:** Describe the steps: normalized each lab using the Normalize API, queried the Knowledge Graph for LOINC code mappings and standardized descriptions.
3. **Paragraph 3 \u2014 Knowledge Graph Value:** The KG provided verified LOINC code mappings and standardized lab descriptions for each laboratory order. Emphasize that these are **curated, clinically-verified relationships** that cannot be derived from text matching \u2014 this is the KG's unique value.
4. **Paragraph 4 \u2014 Recommendation:** Summarize the identified labs with their LOINC codes.

Keep each paragraph concise (2-4 sentences max). Bold key terms."""

print("\u2713  LLM, Tools, and System Prompt ready")
print("   max_tokens=64000, _MAX_TOOL_RESPONSE_CHARS=100000")
print("   Tools: " + str([t.name for t in LAB_ID_TOOLS]))

✓  LLM, Tools, and System Prompt ready
   max_tokens=64000, _MAX_TOOL_RESPONSE_CHARS=100000
   Tools: ['normalize_medical_term', 'identify_lab']


## 2. HTML Renderer + Streaming Engine

CSS ported from `chat_med_refinement.html` + Python functions that render
agent activity into rich HTML panels. Supports **live streaming** via `update_display()`.

In [3]:
def esc(text):
    return html_mod.escape(str(text)) if text else ""

AGENT_CSS = """
<style>
:root {
    --primary: #57039E; --primary-light: #7C0EDD; --primary-dark: #3D0270;
    --accent: #10b981; --danger: #ef4444;
    --bg: #f3f4f6; --bg-card: #ffffff; --text: #1e293b; --text-secondary: #64748b; --border: #e5e7eb;
}
.agent-panel { font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif; color:var(--text); line-height:1.6; margin:1rem 0; }
.agent-panel * { box-sizing:border-box; }
.msg-container { display:flex; gap:0.75rem; margin-bottom:1.25rem; animation:fadeIn 0.3s ease-in; }
@keyframes fadeIn { from{opacity:0;transform:translateY(8px)} to{opacity:1;transform:translateY(0)} }
.msg-avatar { width:36px;height:36px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:1.2rem;flex-shrink:0;background:var(--accent); }
.msg-content { flex:1;background:var(--bg-card);border-radius:12px;padding:1rem;box-shadow:0 1px 2px rgba(0,0,0,0.05); }
.msg-header { display:flex;justify-content:space-between;align-items:center;margin-bottom:0.5rem; }
.msg-sender { font-weight:600;color:var(--text);font-size:0.9rem; }
.msg-time { font-size:0.75rem;color:var(--text-secondary); }
.msg-user .msg-avatar { background:var(--primary); }
.msg-user .msg-content { background:var(--primary);color:#fff; }
.msg-user .msg-sender,.msg-user .msg-time { color:rgba(255,255,255,0.8); }
.reasoning-steps { margin-bottom:1rem;border:1px solid var(--border);border-radius:10px;overflow:hidden;background:var(--bg); }
.reasoning-hdr { display:flex;justify-content:space-between;align-items:center;padding:0.6rem 1rem;background:#f0e7ff;cursor:pointer;user-select:none;border-bottom:1px solid var(--border); }
.reasoning-hdr h4 { font-size:0.88rem;color:var(--primary-dark);margin:0; }
.step-count { font-size:0.68rem;font-weight:600;padding:0.1rem 0.45rem;border-radius:999px;background:var(--primary);color:#fff; }
.toggle-icon { font-size:0.7rem;color:var(--text-secondary);transition:transform 0.2s; }
.reasoning-body { padding:0.6rem;display:flex;flex-direction:column;gap:0.6rem; }
.step { padding:0.7rem 0.85rem;background:var(--bg-card);border-radius:8px;position:relative; }
.step-reasoning { border-left:4px solid #f59e0b;background:linear-gradient(135deg,#fffbeb,var(--bg-card)); }
.step-tool { border-left:4px solid #6d28d9;background:linear-gradient(135deg,#ede9fe,#f5f3ff);box-shadow:0 1px 3px rgba(109,40,217,0.1); }
.step-tool.normalize { border-left-color:#0891b2;background:linear-gradient(135deg,#ecfeff,var(--bg-card));box-shadow:none; }
.step-result { border-left:4px solid var(--accent); }
.step-badge { position:absolute;top:0.5rem;right:0.6rem;color:#fff;font-size:0.63rem;font-weight:700;padding:0.1rem 0.4rem;border-radius:999px;text-transform:uppercase; }
.badge-kg { background:#7c3aed; }
.badge-normalize { background:#0891b2; }
.badge-done { background:var(--accent); }
.step-hdr { display:flex;align-items:center;gap:0.4rem;margin-bottom:0.25rem;font-weight:600;font-size:0.88rem; }
.step-icon { width:22px;height:22px;border-radius:50%;display:inline-flex;align-items:center;justify-content:center;font-size:0.75rem;flex-shrink:0; }
.icon-tool { background:#ede9fe;color:#7c3aed; }
.icon-normalize { background:#cffafe;color:#0e7490; }
.icon-result { background:#d1fae5;color:#065f46; }
.icon-reasoning { background:#fef3c7;color:#92400e; }
.step-detail { font-size:0.83rem;color:var(--text-secondary);padding-left:1.8rem; }
.step-reasoning .step-title { color:#92400e; }
.step-reasoning .reasoning-text { font-size:0.83rem;color:var(--text);line-height:1.5;padding-left:1.8rem; }
.step-tool.normalize .step-title { color:#0e7490; }
.raw-toggle { margin-top:0.5rem;padding-top:0.5rem;border-top:1px solid var(--border); }
.raw-toggle summary { font-size:0.75rem;font-weight:600;color:#7c3aed;cursor:pointer;padding:0.3rem 0.5rem;border-radius:6px; }
.raw-toggle summary:hover { background:rgba(87,3,158,0.05); }
.raw-toggle pre { margin:0.4rem 0 0;background:#f8f9fa;border:1px solid #e2e8f0;border-radius:6px;padding:0.6rem;max-height:300px;overflow-y:auto;font-size:0.72rem;white-space:pre-wrap;word-wrap:break-word; }
.final-answer { margin-top:0.5rem; }
.final-label { font-size:0.73rem;font-weight:700;text-transform:uppercase;letter-spacing:0.05em;color:var(--accent);margin-bottom:0.3rem;padding-bottom:0.3rem;border-bottom:2px solid var(--accent); }
.final-body { font-size:0.95rem;line-height:1.7; }
.final-body strong { color:var(--primary); }
.final-body code { background:#f1f5f9;padding:0.1rem 0.35rem;border-radius:4px;font-size:0.85em; }
.streaming-indicator { display:flex;align-items:center;gap:0.35rem;padding:0.75rem 1rem;color:var(--primary);font-weight:600;font-size:0.88rem;background:linear-gradient(135deg,rgba(87,3,158,0.04),rgba(124,14,221,0.06));border-radius:8px;margin-top:0.5rem; }
.dot-pulse { display:inline-block;width:8px;height:8px;border-radius:50%;background:var(--primary);animation:pulse 1.4s ease-in-out infinite; }
.dot-pulse:nth-child(2) { animation-delay:0.2s; }
.dot-pulse:nth-child(3) { animation-delay:0.4s; }
@keyframes pulse { 0%,100%{opacity:0.2;transform:scale(0.7)} 50%{opacity:1;transform:scale(1.2)} }
</style>
"""

TOOL_NAMES = {
    "normalize_medical_term": "Normalize Medical Term",
    "identify_lab": "GraphQL: Identify Lab (KG Procedure Domain)",
}

def is_normalize(tool_name):
    return tool_name == "normalize_medical_term"

def _loinc_display(loinc_list):
    """Extract LOINC code strings from list — handles both string and object formats."""
    if not loinc_list:
        return ""
    codes = []
    for item in loinc_list:
        if isinstance(item, str):
            codes.append(item)
        elif isinstance(item, dict) and item.get("code"):
            codes.append(item["code"])
    return ", ".join(codes)

def fmt_tool_input(tool_name, args):
    if tool_name == "normalize_medical_term":
        order = args.get("order_name", "")
        order_str = ' (order: <strong>' + esc(order) + '</strong>)' if order else ''
        return (
            'Normalizing <strong>"' + esc(args.get("input_term", ""))
            + '"</strong> in domain <strong>' + esc(args.get("domain", "Lab")) + '</strong>'
            + order_str
        )
    elif tool_name == "identify_lab":
        return (
            'Querying KG procedure domain for lab code <strong>'
            + esc(args.get("lab_code", "")) + '</strong> — getting narrower concepts & LOINC codes'
        )
    return '<pre>' + esc(json.dumps(args, indent=2)) + '</pre>'

def _make_table(headers, rows, max_height="250px", col_widths=None):
    """Build a div-based grid table that bypasses Jupyter's table CSS entirely."""
    n = len(headers)
    if not col_widths:
        col_widths = [str(round(100 / n)) + "%" for _ in range(n)]
    grid_cols = " ".join(col_widths)

    t = '<div style="overflow-x:auto;margin-top:0.3rem;max-height:' + max_height + ';overflow-y:auto;border:1px solid #e5e7eb;border-radius:6px;">'
    t += '<div style="display:grid;grid-template-columns:' + grid_cols + ';background:#57039E;">'
    for h in headers:
        t += '<div style="color:#fff;font-weight:600;text-align:left;padding:0.4rem 0.6rem;white-space:nowrap;font-size:0.8rem;">' + h + '</div>'
    t += '</div>'
    for idx, row in enumerate(rows):
        bg = "#f8fafc" if idx % 2 == 1 else "#ffffff"
        t += '<div style="display:grid;grid-template-columns:' + grid_cols + ';border-bottom:1px solid #e5e7eb;background:' + bg + ';">'
        for cell in row:
            t += '<div style="padding:0.35rem 0.6rem;color:#1e293b;font-size:0.8rem;overflow:hidden;text-overflow:ellipsis;">' + cell + '</div>'
        t += '</div>'
    t += '</div>'
    return t

def fmt_tool_result_html(tool_name, result_str):
    try:
        data = json.loads(result_str) if isinstance(result_str, str) else result_str
    except (json.JSONDecodeError, TypeError):
        return '<pre>' + esc(str(result_str)[:500]) + '</pre>'

    html_out = ""
    if tool_name == "normalize_medical_term" and isinstance(data, dict) and data.get("results"):
        matches = data["results"][0].get("matches", []) if data["results"] else []
        term = data["results"][0].get("input_term", "") if data["results"] else ""
        html_out += '<div style="font-size:0.83rem;color:var(--text-secondary);margin-bottom:0.3rem;">'
        html_out += str(len(matches)) + ' result(s) for "<strong>' + esc(term) + '</strong>"</div>'
        rows = []
        for m in matches[:5]:
            code_val = m.get("default_lexical_code", m.get("lexical_code", ""))
            loinc = _loinc_display(m.get("loinc_codes", []))
            rows.append([
                '<strong>' + esc(m.get("title", "")) + '</strong>',
                esc(code_val),
                "{:.3f}".format(m.get("score", 0)),
                esc(loinc) if loinc else "—",
            ])
        html_out += _make_table(["IMO Description", "Lexical Code", "Score", "LOINC Codes"], rows,
                                col_widths=["40%", "25%", "15%", "20%"])

    elif tool_name == "identify_lab" and isinstance(data, dict) and data.get("lab"):
        lab = data["lab"]
        root_loinc = data.get("loinc_codes", [])
        narrower = data.get("narrower_concepts", [])
        html_out += '<div style="font-size:0.83rem;color:var(--text-secondary);">'
        html_out += 'Lab: <strong>' + esc(lab.get("title", "")) + '</strong>'
        html_out += ' (code: ' + esc(lab.get("code", "")) + ')</div>'
        if root_loinc:
            html_out += '<div style="font-size:0.8rem;margin-top:0.2rem;">Root LOINC: <strong>' + esc(", ".join(root_loinc)) + '</strong></div>'
        if narrower:
            html_out += '<div style="margin-top:0.4rem;"><strong>Narrower Concepts ('
            html_out += str(len(narrower)) + '):</strong></div>'
            rows = []
            for n in narrower:
                loinc_str = ", ".join(n.get("loinc_codes", []))
                rows.append([esc(n.get("title", "")), esc(n.get("code", "")), esc(loinc_str) if loinc_str else "—"])
            html_out += _make_table(["Title", "Code", "LOINC"], rows, "250px",
                                    col_widths=["50%", "25%", "25%"])
        elif not narrower:
            html_out += '<div style="font-size:0.8rem;margin-top:0.3rem;color:#64748b;">No narrower concepts returned.</div>'

    else:
        html_out += '<pre style="max-height:200px;overflow:auto;font-size:0.72rem;">'
        html_out += esc(json.dumps(data, indent=2)[:1000]) + '</pre>'

    raw_json = json.dumps(data, indent=2)
    html_out += '<details class="raw-toggle"><summary>View Response JSON</summary>'
    html_out += '<pre>' + esc(raw_json[:3000]) + '</pre></details>'
    return html_out

def fmt_markdown_to_html(text):
    text = re.sub(r'\[SUMMARY\][\s\S]*?\[/SUMMARY\]', '', text).strip()
    lines = text.split('\n')
    result = []
    in_table = False
    table_rows = []
    for line in lines:
        stripped = line.strip()
        if stripped.startswith('|') and stripped.endswith('|'):
            if re.match(r'^\|[\s\-:|]+\|$', stripped):
                continue
            if not in_table:
                in_table = True
                table_rows = []
            table_rows.append([c.strip() for c in stripped.split('|')[1:-1]])
        else:
            if in_table:
                result.append(_build_table(table_rows))
                in_table = False
            result.append(stripped)
    if in_table:
        result.append(_build_table(table_rows))
    f = '\n'.join(result)
    f = re.sub(r'^#### (.+)$', r'<h4>\1</h4>', f, flags=re.M)
    f = re.sub(r'^### (.+)$', r'<h4>\1</h4>', f, flags=re.M)
    f = re.sub(r'^## (.+)$', r'<h3>\1</h3>', f, flags=re.M)
    f = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', f)
    f = re.sub(r'\*(.+?)\*', r'<em>\1</em>', f)
    f = re.sub(r'`(.+?)`', r'<code>\1</code>', f)
    f = f.replace('\n\n', '</p><p>').replace('\n', '<br>')
    return '<p>' + f + '</p>'

def _build_table(rows):
    if not rows:
        return ''
    headers = rows[0]
    data_rows = rows[1:]
    n = len(headers)
    if n == 2:
        widths = ["65%", "35%"]
    elif n == 3:
        widths = ["50%", "30%", "20%"]
    elif n == 4:
        widths = ["35%", "25%", "20%", "20%"]
    elif n == 5:
        widths = ["25%", "20%", "20%", "20%", "15%"]
    elif n == 6:
        widths = ["20%", "18%", "15%", "17%", "15%", "15%"]
    else:
        widths = None
    return _make_table(headers, data_rows, "300px", col_widths=widths)

def _streaming_dots(msg="Agent is working..."):
    return (
        '<div class="streaming-indicator">'
        '<span class="dot-pulse"></span><span class="dot-pulse"></span><span class="dot-pulse"></span>'
        '<span style="margin-left:0.3rem;">' + esc(msg) + '</span>'
        '</div>'
    )

def render_agent_run(activity, final_text, label="Lab Identification Results", streaming=False):
    normalize_n = sum(1 for a in activity if a["type"] == "tool_call" and is_normalize(a["tool"]))
    kg_n = sum(1 for a in activity if a["type"] == "tool_call" and not is_normalize(a["tool"]))
    count_parts = []
    if normalize_n:
        count_parts.append(str(normalize_n) + " Normalize")
    if kg_n:
        count_parts.append(str(kg_n) + " KG")
    count_label = " · ".join(count_parts) if count_parts else "0 calls"

    norm_idx = 0
    kg_idx = 0
    steps_html = ""
    for a in activity:
        if a["type"] == "reasoning" and a.get("text", "").strip():
            steps_html += (
                '<div class="step step-reasoning">'
                '<div class="step-hdr"><span class="step-icon icon-reasoning">\U0001f9e0</span>'
                '<span class="step-title">Agent Reasoning</span></div>'
                '<div class="reasoning-text">' + fmt_markdown_to_html(a["text"]) + '</div>'
                '</div>'
            )
        elif a["type"] == "tool_call":
            tool_name = a["tool"]
            norm = is_normalize(tool_name)
            if norm:
                norm_idx += 1
                idx = norm_idx
            else:
                kg_idx += 1
                idx = kg_idx
            cls = "step-tool normalize" if norm else "step-tool"
            badge_cls = "badge-normalize" if norm else "badge-kg"
            badge_lbl = "Normalize" if norm else "KG Call"
            icon_cls = "icon-normalize" if norm else "icon-tool"
            icon = "⚙" if norm else "\U0001f50d"
            display_name = TOOL_NAMES.get(tool_name, tool_name)
            detail = fmt_tool_input(tool_name, a.get("args", {}))
            steps_html += (
                '<div class="step ' + cls + '">'
                '<span class="step-badge ' + badge_cls + '">' + badge_lbl + ' ' + str(idx) + '</span>'
                '<div class="step-hdr"><span class="step-icon ' + icon_cls + '">' + icon + '</span>'
                '<span class="step-title">' + esc(display_name) + '</span>'
                '<span class="step-badge badge-done" style="position:static;margin-left:auto;">Done</span></div>'
                '<div class="step-detail">' + detail + '</div>'
                '</div>'
            )
        elif a["type"] == "tool_result":
            tool_name = a["tool"]
            display_name = TOOL_NAMES.get(tool_name, tool_name)
            result_html = fmt_tool_result_html(tool_name, a.get("result", ""))
            steps_html += (
                '<div class="step step-result">'
                '<div class="step-hdr"><span class="step-icon icon-result">☰</span>'
                '<span class="step-title">Results from ' + esc(display_name) + '</span></div>'
                '<div>' + result_html + '</div>'
                '</div>'
            )

    if final_text:
        bottom = (
            '<div class="final-answer">'
            '<div class="final-label">' + esc(label) + '</div>'
            '<div class="final-body">' + fmt_markdown_to_html(final_text) + '</div>'
            '</div>'
        )
    elif streaming:
        bottom = _streaming_dots("Agent is working...")
    else:
        bottom = ""

    if not activity and streaming:
        reasoning_section = _streaming_dots("Agent is thinking...")
    else:
        reasoning_section = (
            '<div class="reasoning-steps">'
            '<div class="reasoning-hdr" onclick="var b=this.parentElement.querySelector('
            "'.reasoning-body');b.style.display=b.style.display==='none'?'flex':'none'" + '">'
            '<h4>Agent Reasoning &amp; API Calls</h4>'
            '<span class="step-count">' + count_label + '</span>'
            '<span class="toggle-icon">▼</span>'
            '</div>'
            '<div class="reasoning-body">' + steps_html + '</div>'
            '</div>'
        )

    return (
        AGENT_CSS
        + '<div class="agent-panel">'
        '<div class="msg-container">'
        '<div class="msg-avatar">\U0001f916</div>'
        '<div class="msg-content">'
        '<div class="msg-header">'
        '<span class="msg-sender">Lab Identification Agent</span>'
        '</div>'
        + reasoning_section
        + bottom
        + '</div></div></div>'
    )

print("✓  HTML renderer + streaming engine ready")

✓  HTML renderer + streaming engine ready


## 3. Agent + Streaming Runner

Creates the ReAct agent and provides `stream_agent()` — streams via
`agent.stream(stream_mode='values')` and live-updates the HTML display
after every agent/tool step.

In [4]:
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(llm, tools=LAB_ID_TOOLS, prompt=SYSTEM_PROMPT)

# --- Conversation history management (mirrors route's _conversations) ---
_conversations = {}
_NOTE_MIN_WORDS = 30

def _extract_text(content):
    """Extract text from msg.content (str or list of content blocks)."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, str):
                parts.append(block)
            elif isinstance(block, dict) and block.get("type") == "text":
                parts.append(block.get("text", ""))
        return "\n".join(parts)
    return str(content) if content else ""

def _build_activity(messages, skip_count=0):
    """Parse messages into (activity_list, final_text). Skip first skip_count messages."""
    activity = []
    final_text = ""
    for msg in messages[skip_count:]:
        role = msg.type if hasattr(msg, 'type') else 'unknown'
        if role == 'ai':
            text = _extract_text(msg.content)
            if text.strip():
                if hasattr(msg, 'tool_calls') and msg.tool_calls:
                    activity.append({"type": "reasoning", "text": text})
                else:
                    final_text = text
            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                for tc in msg.tool_calls:
                    activity.append({"type": "tool_call", "tool": tc["name"], "args": tc.get("args", {})})
        elif role == 'tool':
            content = msg.content if isinstance(msg.content, str) else json.dumps(msg.content)
            activity.append({"type": "tool_result", "tool": msg.name, "result": content})
    return activity, final_text

def stream_agent(user_message, display_id, session_id="notebook",
                 label="Lab Identification Results", recursion_limit=100):
    """Stream agent execution with live HTML updates and conversation history.

    Mirrors the route's prompt context management:
    - New clinical notes (>=30 words) clear conversation history
    - Keeps only last 2 messages to stay within context limits
    - Stores assistant response back into conversation history

    Returns (activity, final_text, all_messages).
    """
    update_display(HTML(render_agent_run([], "", label=label, streaming=True)), display_id=display_id)

    session_key = session_id + "_lab-id"

    is_new_clinical_note = len(user_message.split()) >= _NOTE_MIN_WORDS
    if is_new_clinical_note and session_key in _conversations and _conversations[session_key]:
        _conversations[session_key] = []

    messages = _conversations.get(session_key, [])
    if len(messages) > 2:
        messages = messages[-2:]
    messages.append({"role": "user", "content": user_message})

    skip_count = len(messages)

    try:
        last_state = None
        for state in agent.stream(
            {"messages": messages},
            stream_mode="values",
            config={"recursion_limit": recursion_limit},
        ):
            last_state = state
            activity, final_text = _build_activity(state["messages"], skip_count=skip_count)
            html = render_agent_run(activity, final_text, label=label, streaming=not bool(final_text))
            update_display(HTML(html), display_id=display_id)

        all_messages = last_state["messages"] if last_state else messages
        activity, final_text = _build_activity(all_messages, skip_count=skip_count)
        update_display(HTML(render_agent_run(activity, final_text, label=label, streaming=False)), display_id=display_id)

        if final_text:
            messages.append({"role": "assistant", "content": final_text})
        _conversations[session_key] = messages

        return activity, final_text, all_messages

    except Exception as e:
        error_msg = "Agent execution failed: " + str(e)
        print("  ✗ " + error_msg)
        import traceback
        traceback.print_exc()
        error_html = (
            '<div style="padding:1rem;background:#fef2f2;border:1px solid #fecaca;border-radius:8px;'
            'color:#991b1b;font-size:0.9rem;margin-top:0.5rem;">'
            '<strong>Error:</strong> ' + esc(error_msg) + '</div>'
        )
        update_display(HTML(AGENT_CSS + error_html), display_id=display_id)
        return [], "", messages

print("✓  Agent + streaming runner ready (with conversation history management)")

✓  Agent + streaming runner ready (with conversation history management)


D:\Users\nthonte\AppData\Local\Temp\1\ipykernel_16304\4019730314.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=LAB_ID_TOOLS, prompt=SYSTEM_PROMPT)


## 4. Clinical Note (Lab Orders)

In [5]:
LAB_QUERY = """Patient Profile
Patient: Adult patient
Reason for evaluation: Follow-up of chronic kidney disease and routine monitoring of renal function
Relevant history: Known chronic kidney disease with progressively reduced renal function. No acute symptoms reported at the time of evaluation.
Assessment
The patient has laboratory findings consistent with chronic kidney disease (CKD) stage 3b, characterized by an estimated glomerular filtration rate (eGFR) of 30\u201344 mL/min/1.73 m\u00b2. Additional testing was obtained to assess renal function, albuminuria, electrolyte status, anemia, and CKD-related mineral and bone abnormalities.
Laboratory Orders
The following laboratory studies were ordered:
1. Comprehensive metabolic panel (CMP)
2. Estimated glomerular filtration rate (eGFR)
3. Urinalysis with microscopic examination
4. Urine albumin-to-creatinine ratio (UACR)
5. Complete blood count (CBC)
6. Serum phosphorus
7. Intact parathyroid hormone (PTH)
8. 25-hydroxy vitamin D
9. Serum magnesium
10. Hemoglobin A1c, as clinically indicated
11. Lipid panel"""

user_html = (
    AGENT_CSS
    + '<div class="agent-panel"><div class="msg-container msg-user">'
    + '<div class="msg-avatar">\U0001f464</div>'
    + '<div class="msg-content">'
    + '<div class="msg-header"><span class="msg-sender">You</span></div>'
    + '<div style="font-size:1.05rem;white-space:pre-wrap;">' + esc(LAB_QUERY) + '</div>'
    + '</div></div></div>'
)
display(HTML(user_html))

## 5. Run Agent — Lab Identification (Streaming)

Sends the clinical note and streams the full agent workflow:
1. **Extract & Classify** labs from the note (ORDER vs COMPONENT)
2. **Normalize** each lab via IMO Precision Normalize API (domain="Lab")
3. **Query KG** for each lab's narrower concepts and LOINC codes
4. **Select best** narrower concept based on clinical context
5. **Present results** — IMO description, IMO code, and LOINC codes for each lab

In [6]:
stream_id = "labid_" + uuid.uuid4().hex[:6]
display(HTML(render_agent_run([], "", label="Lab Identification Results", streaming=True)), display_id=stream_id)

activity, final_text, all_messages = stream_agent(
    LAB_QUERY,
    display_id=stream_id,
    session_id="notebook",
    label="Lab Identification Results",
    recursion_limit=100,
)

tc_count = sum(1 for a in activity if a['type'] == 'tool_call')
rs_count = sum(1 for a in activity if a['type'] == 'reasoning')
print("✓ Complete. " + str(tc_count) + " tool calls, " + str(rs_count) + " reasoning steps.")

✓ Complete. 22 tool calls, 2 reasoning steps.
